# Inferring network from single-cell data with CausFate

## Structure learning

### Load packages

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(foreach)
    library(dplyr)
    library(tidyr)
    library(bnlearn)
    library(Matrix)
    library(space)
    library(infotheo)
    library(igraph)
    library(graph)
    library(Rgraphviz)
    library(tibble)
})

library(causfate)

Setting plotting options for jupyter notebooks. These settings are not necessary when running the code in an R environment outside of jupyter notebooks.

In [ ]:
options(repr.plot.format = "png", repr.plot.width = 5.5, repr.plot.height = 5.5)
options(jupyter.plot_mimetypes = "image/png")

### Load data

In [ ]:
data("HMR")
head(HMR)

In [ ]:
dim(HMR)

### Find variable genes

Note that `gem2mem` is used to transform the single-cell data to cell type-specific gene profiles.

In [ ]:
HMR <- FindVariableFeatures(HMR, selection.method = "vst", nfeatures = nrow(HMR))
glist <- VariableFeatures(HMR)

meta <- HMR@meta.data[["celltype"]] %>% setNames(colnames(HMR))
ctypes <- c("LTHSC", "MPP", "CMP", "LMPP", "MEP", "GMP")
pseudobulk_expmat <- gem2mem(data.frame(HMR@assays$RNA@data), meta, "mean")

### BMA process

`BNLearning` samples the data and performs the inner BMA process, giving out multiple one-shot DAGs. `combineDAGsmpl` combines those DAGs before pruning.

In [ ]:
dag_smpl <- BNLearning(HMR[glist[1:5000]],
  resampling_fraction = 0.2, # Fraction of cells used in each resampled dataset
  N_smpl = 40, params = seq(0.05, 0.3, 0.05),
  root = "LTHSC", mode = "single_cell", ncores = 1,
  dagMethod = "hc", ugMethod = "cmi2ni", seed = 42
)
net <- combineDAGsmpl(dag_smpl, Emin = 1, Emax = 7, ncores = 1)

# Let us take a glimpse of the structure now
net_mat <- net %>% df2mat() %>% .[ctypes, ctypes]
net_mat

First, very unlikely edges are trimmed if the occurrence is less than 2. Then, `rmCyc` removes cycles, preserving the more possible edges.

In [ ]:
net <- (net_mat * (net_mat > 2)) %>%
  mat2df() %>%
  rmCyc()
e <- df2bn(net, ctypes, plot = TRUE)

`trimDAG` performs the final local pruning step.

In [ ]:
dag_struc <- trimDAG(pseudobulk_expmat[glist[1:4000], ], e, min_arc = 2, max_arc = 2, threshold_value = 1, plot = FALSE)

In [ ]:
node_col <- c(
  "LTHSC" = "#9A9A99",
  "MPP" = "#B69D75",
  "CMP" = "#FCED21",
  "LMPP" = "#63BDC5",
  "MEP" = "#D59546",
  "GMP" = "#A4C362"
)
ref_net <- bnlearn::model2network("[LTHSC][MPP|LTHSC][CMP|MPP][LMPP|MPP][MEP|CMP][GMP|CMP:LMPP]")
colorEdges(dag_struc, ref_net, 
           node_col = node_col, 
           fontsize = 10, 
           node_size = 2,
           sep = 0.8)

## Calculate gene-wise causal effects

Generate indexes for bootstrapping.

In [ ]:
set.seed(42)
index <- bootstrap_index(
  meta = meta,
  bootstrap_times = 10,
  resampling_fraction = 0.3 # Fraction of cells sampled from each cell state
)

This updated example uses zeroing, the default in `PerturbResult` (`deletion = FALSE`, `perturb_ratio = 0`). Ratios strictly between 0 and 1 model downward scaling (knockdown), ratios above 1 upward scaling (knockup), and 1 leaves the values unchanged. For example, change `perturb_ratio` below to 0.5 or 2. Use `deletion = TRUE` for feature deletion. These operations act on the supplied values.

Version note: the main HMR analysis in the manuscript used permutation; graded scaling was not used in the reported analyses. This zeroing tutorial is a software demonstration, not a reproduction of the original HMR gene ranking.

`EffectMatrix(..., dist_metric = 'diff_mean')` retains signed differences summed across runs; `diffScore(abs = TRUE)` takes absolute edge effects before scoring. Other case-insensitive metrics are W1, W2, energy and mmd (e.g. `EffectMatrix(perturbRes, dist_metric = 'w1')` or `dist_metric = 'MMD'`). The legacy `mode = 'mean'` remains supported.

In [ ]:
perturbRes <- PerturbResult(
  net_struc = dag_struc,
  data = HMR,
  meta = meta,
  index = index,
  n_sample = 10,
  mode = "single_cell", # GRN-free single-feature perturbation
  deletion = FALSE,
  perturb_ratio = 0, # Knockout by zeroing
  ncores = 1
)
effMat <- EffectMatrix(perturbRes, dist_metric = "diff_mean")

Score and rank genes with `diffScore` and `dsRank`. Here all edges are selected and overall effects to the network are calculated. A subset of edges can be chosen to study for specific differentiation processes.

In [ ]:
edgeSet <- setEdges(fromSet = ctypes, toSet = ctypes) # Calculate for all edges
effectScore <- diffScore(effMat, edgeSet)
geneList <- dsRank(effectScore)
head(geneList)